# Auditing a model on `heart_cleveland`

A full isitfair audit of a logistic regression fitted to the public
`heart_cleveland` dataset: discrimination, calibration, decision curves,
Hosmer-Lemeshow, estimability, a recalibration ladder, mitigation and an
HTML report.

Before running, build the prediction table once from the repository root:

    python -I extra_examples/datasets.py heart_cleveland

`B = 50` bootstrap replicates keep the run short; use 2,000 for results you
intend to report.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "extra_examples" / "datasets.py").exists():
        HERE = _c / "extra_examples"
        break
sys.path.insert(0, str(HERE))
import datasets  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing  # noqa: E402

NAME = "heart_cleveland"
B = 50
SEED = 42
META = datasets.meta(NAME)
THRESHOLD = META["threshold"]
ATTRS = datasets.ATTRS[NAME]
print(f"isitfair {isitfair.__version__}")
print(f"{NAME}: outcome {META['outcome']}; threshold {THRESHOLD}; B = {B}")

isitfair 0.4.0
heart_cleveland: outcome num > 0 (any angiographic disease); threshold 0.5; B = 50


## 1. Load the predictions

One row per test-set patient or person: the outcome, the model's predicted
probability, and the attributes to audit.

In [2]:
df = datasets.load(NAME)
print(f"n = {len(df)}, prevalence = {df['y_true'].mean():.3f}")
for a in ATTRS:
    print(f"{a}: {df[a].nunique()} levels, {int(df[a].isna().sum())} missing")

n = 91, prevalence = 0.462
sex: 2 levels, 0 missing
age_group: 4 levels, 0 missing


isitfair refuses missing subgroup labels rather than silently dropping those
rows. If any attribute has missing labels, this cell shows the error and then
recodes them to an explicit `"Unknown"` level.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    try:
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    except ValueError as exc:
        print("ValueError:", exc)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

recoded to 'Unknown': none


## 2. Split off a calibration half for mitigation

The test half is split 50/50, stratified on the outcome. Mitigation
calibrators are fitted on one half and every audit runs on the other, so no
row is scored by a calibrator fitted on it.

In [4]:
i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                 stratify=df["y_true"])
cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
# mitigate() rejects audited levels that the calibration half never saw; drop
# such rows (for example diabetes130's single "Unknown/Invalid" gender).
unseen = np.zeros(len(eval_df), bool)
for a in ATTRS:
    unseen |= ~eval_df[a].isin(set(cal_df[a]))
if unseen.any():
    print(f"dropping {int(unseen.sum())} audited row(s) with levels absent from the calibration half")
    eval_df = eval_df[~unseen].reset_index(drop=True)
print(f"calibration n = {len(cal_df)}, audit n = {len(eval_df)}")

calibration n = 45, audit n = 46


## 3. Construct the audit

Each attribute is audited on its own, and the first two are also crossed
(passing `None` for a tuple key asks isitfair to build the intersection).

In [5]:
y = eval_df["y_true"].to_numpy(dtype=float)
p = eval_df["y_proba"].to_numpy(dtype=float)
marginals = {a: eval_df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None

audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                      n_bootstrap=B, random_state=SEED)
audit

FairnessAudit(n=46, events=21, threshold=0.5, n_bootstrap=50, random_state=42, attributes: sex (2), age_group (4), sex × age_group (8))

## 4. Discrimination

In [6]:
disc = audit.discrimination()
disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                              "auroc_ci_low", "auroc_ci_high"]]

,attribute,subgroup,n,n_events,auroc,auroc_ci_low,auroc_ci_high
0,sex,Overall,46.0,21.0,0.935238,NaN,NaN
1,sex,female,20.0,6.0,1.000000,NaN,NaN
2,sex,male,26.0,15.0,0.878788,NaN,NaN
7,age_group,Overall,46.0,21.0,0.935238,NaN,NaN
8,age_group,45-54,10.0,2.0,1.000000,NaN,NaN
9,age_group,55-64,20.0,12.0,0.864583,NaN,NaN
10,age_group,65+,8.0,4.0,1.000000,NaN,NaN
11,age_group,<45,8.0,3.0,1.000000,NaN,NaN
16,sex × age_group,Overall,46.0,21.0,0.935238,NaN,NaN
17,sex × age_group,female / 45-54,4.0,0.0,NaN,NaN,NaN


## 5. Calibration

In [7]:
cal = audit.calibration()
cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                            "calibration_slope", "brier", "ece", "ici"]]

,attribute,subgroup,n,calibration_intercept,calibration_slope,brier,ece,ici
0,sex,Overall,46.0,-0.770234,1.222992,0.106272,0.135877,0.089877
1,sex,female,20.0,-1.146819,22.463007,0.067301,0.199720,0.172723
2,sex,male,26.0,-0.480360,0.803768,0.136250,0.151388,0.064893
5,age_group,Overall,46.0,-0.770234,1.222992,0.106272,0.135877,0.089877
6,age_group,45-54,10.0,-2.420576,14.189740,0.121794,0.252801,0.219979
7,age_group,55-64,20.0,-0.280863,0.861855,0.101868,0.162432,0.107901
8,age_group,65+,8.0,-1.201421,108.098427,0.132905,0.270796,0.160864
9,age_group,<45,8.0,0.168714,91.052232,0.071246,0.210129,0.133326
12,sex × age_group,Overall,46.0,-0.770234,1.222992,0.106272,0.135877,0.089877
13,sex × age_group,female / 45-54,4.0,NaN,NaN,0.090166,0.245278,0.245278


In [8]:
curves = audit.calibration_curves()
roc = audit.roc_curves()
print("calibration curves:", curves.shape, "| ROC curves:", roc.shape)

calibration curves: (1700, 7) | ROC curves: (1700, 7)


## 6. Decision curve: gaps in net benefit between subgroups

In [9]:
dc = audit.decision_curve()
dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]]

,attribute,gap_value,gap_ci_low,gap_ci_high
60,sex,0.311538,NaN,NaN
161,age_group,0.483333,NaN,NaN
342,sex × age_group,0.837838,NaN,NaN


## 7. Estimability

Cells with too few events for a metric are flagged rather than reported.

In [10]:
est = audit.estimability()
print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

191 of 208 cell-metrics not reportable


## 8. Hosmer-Lemeshow

In [11]:
hl = audit.hosmer_lemeshow()
hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                          "hl_p_value", "n_bins_used"]]

,attribute,subgroup,n,hl_statistic,hl_df,hl_p_value,n_bins_used
0,sex,Overall,46.0,12.151093,10.0,0.275072,10.0
1,sex,female,20.0,6.435613,10.0,0.777434,10.0
2,sex,male,26.0,20.792276,10.0,0.022590,10.0
4,age_group,Overall,46.0,12.151093,10.0,0.275072,10.0
5,age_group,45-54,10.0,5.990160,10.0,0.816089,10.0
6,age_group,55-64,20.0,16.156815,10.0,0.095231,10.0
7,age_group,65+,8.0,4.786120,8.0,0.780172,8.0
8,age_group,<45,8.0,2.677993,8.0,0.952914,8.0
10,sex × age_group,Overall,46.0,12.151093,10.0,0.275072,10.0
11,sex × age_group,female / 45-54,4.0,NaN,NaN,NaN,4.0


## 9. Intersection cells

In [12]:
audit.cell_diagnostics() if INTERSECTION else "single attribute; no intersection"

,attribute,subgroup,n,n_events,shrunk,marginal_rate,cell_rate,shrinkage_factor
0,sex × age_group,female / 45-54,4,0,True,0.300000,0.000000,0.663158
1,sex × age_group,female / 55-64,9,4,True,0.300000,0.444444,0.466667
2,sex × age_group,female / 65+,4,1,True,0.300000,0.250000,0.663158
3,sex × age_group,female / <45,3,1,True,0.300000,0.333333,0.724138
4,sex × age_group,male / 45-54,6,2,True,0.576923,0.333333,0.567568
5,sex × age_group,male / 55-64,11,8,True,0.576923,0.727273,0.417219
6,sex × age_group,male / 65+,4,3,True,0.576923,0.750000,0.663158
7,sex × age_group,male / <45,5,2,True,0.576923,0.400000,0.611650


## 10. Recalibration ladder on the attribute with the largest intercept gap

A level with no events at all cannot be recalibrated, and isitfair says so.
In that case the ladder moves on to the next attribute.

In [13]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]
ladder = None
for by in RANKED:
    try:
        ladder = audit.recalibration_ladder(by=by)
        print("ladder by", by)
        break
    except ValueError as exc:
        print(f"{by}: {exc}")
ladder

ladder by age_group


,rung,attribute,subgroup,n,n_events,calibration_intercept,intercept_ci_low,intercept_ci_high,calibration_slope,slope_ci_low,...,brier_ci_low,brier_ci_high,ece,ici,net_benefit,gap_metric,gap_value,event_floor,below_floor,not_estimable_final
0,none,age_group,Overall,46.0,21.0,-0.770234,NaN,NaN,1.222992,NaN,...,NaN,NaN,0.135877,0.089877,0.304348,NaN,NaN,25.0,True,True
1,none,age_group,45-54,10.0,2.0,-2.420576,NaN,NaN,14.189740,NaN,...,NaN,NaN,0.252801,0.219979,-0.100000,NaN,NaN,25.0,True,True
2,none,age_group,55-64,20.0,12.0,-0.280863,NaN,NaN,0.861855,NaN,...,NaN,NaN,0.162432,0.107901,0.500000,NaN,NaN,25.0,True,True
3,none,age_group,65+,8.0,4.0,-1.201421,NaN,NaN,108.098427,NaN,...,NaN,NaN,0.270796,0.160864,0.250000,NaN,NaN,25.0,True,True
4,none,age_group,<45,8.0,3.0,0.168714,NaN,NaN,91.052232,NaN,...,NaN,NaN,0.210129,0.133326,0.375000,NaN,NaN,25.0,True,True
5,none,age_group,GAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,max_intercept_gap,2.589290,NaN,NaN,NaN
6,common,age_group,Overall,46.0,21.0,0.052395,NaN,NaN,0.119557,NaN,...,NaN,NaN,0.131506,0.051936,0.347826,NaN,NaN,25.0,True,True
7,common,age_group,45-54,10.0,2.0,-2.645547,NaN,NaN,1.484823,NaN,...,NaN,NaN,0.167216,0.148838,0.100000,NaN,NaN,25.0,True,True
8,common,age_group,55-64,20.0,12.0,0.355233,NaN,NaN,0.082362,NaN,...,NaN,NaN,0.143082,0.138994,0.500000,NaN,NaN,25.0,True,True
9,common,age_group,65+,8.0,4.0,0.401411,NaN,NaN,1.566534,NaN,...,NaN,NaN,0.146429,0.055115,0.375000,NaN,NaN,25.0,True,True


## 11. Mitigation

Group-wise isotonic recalibration, then Wasserstein post-processing, both
fitted on the calibration half.

In [14]:
calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba"].to_numpy(float),
              {a: cal_df[a].to_numpy(object) for a in ATTRS})
mitigated_gr = audit.mitigate(method=GroupRecalibration(method="isotonic"), calibration_data=calib_data)
a0 = ATTRS[0]
keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before, after = audit.calibration(), mitigated_gr[a0].calibration()
print("Before,", a0)
print(before[before.attribute == a0][keep].to_string(index=False))
print("\nAfter group isotonic recalibration,", a0)
print(after[after.attribute == a0][keep].to_string(index=False))

Before, sex
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall              -0.770234           1.222992 0.106272 0.135877
  female              -1.146819          22.463007 0.067301 0.199720
    male              -0.480360           0.803768 0.136250 0.151388
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN

After group isotonic recalibration, sex
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall             -19.981328           0.133180 0.149448 0.142528
  female             -19.999995           0.794779 0.177101 0.237925
    male              -1.531475           0.124766 0.128176 0.123163
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN


In [15]:
mitigated_w = audit.mitigate(method=WassersteinPostprocessing(random_state=SEED), calibration_data=calib_data)
d_after = mitigated_w[a0].discrimination()
d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
    ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]

,gap_metric,gap_value,gap_ci_low,gap_ci_high
3,equal_opportunity_difference,0.133333,NaN,NaN
4,predictive_equality,0.012987,NaN,NaN
5,equalized_odds_gap,0.133333,NaN,NaN
6,statistical_parity,0.115385,NaN,NaN


## 12. HTML report

In [16]:
REPORT = HERE / "reports" / f"{NAME}.html"
REPORT.parent.mkdir(exist_ok=True)
audit.report(out=REPORT, title=f"{NAME}: isitfair audit",
             metadata={"Dataset": NAME, "Model": "LogisticRegression",
                       "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
             mitigation_audits=mitigated_gr)
print("wrote", REPORT.relative_to(HERE.parent))

wrote extra_examples\reports\heart_cleveland.html
